## Sentiment Analysis

In this exercise we use the IMDb-dataset, which we will use to perform a sentiment analysis. The code below assumes that the data is placed in the same folder as this notebook. We see that the reviews are loaded as a pandas dataframe, and print the beginning of the first few reviews.

In [1]:
import numpy as np
import pandas as pd

reviews = pd.read_csv('reviews.txt', header=None)
labels = pd.read_csv('labels.txt', header=None)
Y = (labels=='positive').astype(np.int_)

print(type(reviews))
print(reviews.head())

<class 'pandas.core.frame.DataFrame'>
                                                   0
0  bromwell high is a cartoon comedy . it ran at ...
1  story of a man who has unnatural feelings for ...
2  homelessness  or houselessness as george carli...
3  airport    starts as a brand new luxury    pla...
4  brilliant over  acting by lesley ann warren . ...


**(a)** Split the reviews and labels in test, train and validation sets. The train and validation sets will be used to train your model and tune hyperparameters, the test set will be saved for testing. Use the `CountVectorizer` from `sklearn.feature_extraction.text` to create a Bag-of-Words representation of the reviews. Only use the 10,000 most frequent words (use the `max_features`-parameter of `CountVectorizer`).

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer

reviews.columns = ['text']
Y.columns = ['sentiment']

X_full, X_test, y_full, y_test = train_test_split(reviews, Y, test_size = 0.2, random_state = 42)
X_train, X_val, y_train, y_val = train_test_split(X_full, y_full, test_size = 0.2, random_state = 42)

vectorizer = CountVectorizer(max_features = 10000)

train_vec = vectorizer.fit_transform(X_train['text'])
val_vec = vectorizer.transform(X_val['text'])
test_vec = vectorizer.transform(X_test['text'])

print(vectorizer.vocabulary_)

{'the': 8954, 'michael': 5652, 'keaton': 4894, 'comedy': 1731, 'of': 6163, 'same': 7662, 'title': 9073, 'was': 9680, 'condemned': 1826, 'for': 3518, 'it': 4722, 'um': 9312, 'shoddy': 8007, 'special': 8334, 'effects': 2847, 'but': 1206, 'compared': 1769, 'to': 9079, 'what': 9760, 'screaming': 7764, 'mad': 5378, 'george': 3748, 'up': 9434, 'this': 8995, 'horror': 4301, 'they': 8980, 're': 7125, 'positively': 6732, 'mind': 5687, 'boggling': 979, 'killer': 4937, 'snowman': 8235, 'seems': 7827, 'be': 752, 'made': 5379, 'out': 6276, 'and': 330, 'his': 4216, 'arms': 469, 'look': 5285, 'like': 5190, 'which': 9772, 'probably': 6861, 'were': 9748, 'cast': 1353, 'lays': 5088, 'on': 6203, 'thick': 8981, 'in': 4473, 'parody': 6398, 'dozens': 2667, 'other': 6268, 'much': 5859, 'worse': 9905, 'movies': 5851, 'paul': 6446, 'keith': 4903, 'as': 505, 'town': 9151, 'doctor': 2606, 'is': 4709, 'particularly': 6408, 'memorable': 5603, 'small': 8203, 'hilarious': 4196, 'role': 7537, 'beautiful': 767, 'film'

**(b)** Explore the representation of the reviews. How is a single word represented? How about a whole review?

In [3]:
print(f"index of 'hate' - {vectorizer.vocabulary_.get('hate')}\n") # word

print(train_vec[0]) # review

index of 'hate' - 4081

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 66 stored elements and shape (1, 10000)>
  Coords	Values
  (0, 8954)	5
  (0, 5652)	1
  (0, 4894)	1
  (0, 1731)	2
  (0, 6163)	4
  (0, 7662)	1
  (0, 9073)	1
  (0, 9680)	1
  (0, 1826)	1
  (0, 3518)	2
  (0, 4722)	2
  (0, 9312)	1
  (0, 8007)	1
  (0, 8334)	1
  (0, 2847)	1
  (0, 1206)	2
  (0, 1769)	1
  (0, 9079)	2
  (0, 9760)	1
  (0, 7764)	1
  (0, 5378)	1
  (0, 3748)	1
  (0, 9434)	1
  (0, 8995)	2
  (0, 4301)	1
  :	:
  (0, 9772)	1
  (0, 6861)	1
  (0, 9748)	1
  (0, 1353)	1
  (0, 5088)	1
  (0, 6203)	1
  (0, 8981)	1
  (0, 4473)	2
  (0, 6398)	1
  (0, 2667)	1
  (0, 6268)	1
  (0, 5859)	1
  (0, 9905)	1
  (0, 5851)	1
  (0, 6446)	1
  (0, 4903)	1
  (0, 505)	1
  (0, 9151)	1
  (0, 2606)	1
  (0, 4709)	1
  (0, 6408)	1
  (0, 5603)	1
  (0, 8203)	1
  (0, 4196)	1
  (0, 7537)	1


The corpus is a dictionary. A single word is represented with a position in the corpus. A whole review is represented as a vector where the positions from the corpus are used, and the values at those positions are the amount of times the word shows up in the review.

**(c)** Train a neural network with a single hidden layer on the dataset, tuning the relevant hyperparameters to optimize accuracy. 

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, regularizers
import keras_tuner as kt

def model_builder(hp):
    reg_type = hp.Choice('regularizer', ['l1', 'l2'])
    reg_strength = hp.Choice('reg_strength', values = [1e-4, 1e-3, 1e-2, 1e-1])

    if reg_type == 'l1':
        regularizer = regularizers.l1(reg_strength)
    elif reg_type == 'l2':
        regularizer = regularizers.l2(reg_strength)

    model = tf.keras.Sequential()
    model.add(layers.Dense(
        units = hp.Choice('units', values = [8, 16, 32, 64]),
        activation = hp.Choice('activation', values = ['relu', 'tanh']),
        input_dim = train_vec.shape[1],
        kernel_regularizer = regularizer
    ))
    model.add(layers.Dropout(0.25))
    model.add(layers.Dense(1, activation = 'sigmoid'))

    model.compile(
        optimizer = hp.Choice('optimizer', values = ['SGD','adam', 'adamw']),
        loss = 'binary_crossentropy',
        metrics = 'accuracy'
    )

    return model

In [5]:
tuner = kt.Hyperband(
    model_builder,
    objective = 'val_accuracy',
    max_epochs = 20,
    factor = 3,
    directory = 'my_dir',
    project_name = 'tune_model'
)

early_stop = tf.keras.callbacks.EarlyStopping(monitor = 'val_loss', patience = 3, restore_best_weights = True)
tuner.search(train_vec.toarray(), y_train, epochs = 20, validation_data = (val_vec.toarray(), y_val), callbacks = [early_stop])

best_model = tuner.get_best_models(num_models = 1)[0]
print(tuner.get_best_hyperparameters()[0].values)

Trial 22 Complete [00h 00m 00s]

Best val_accuracy So Far: 0.8920000195503235
Total elapsed time: 00h 00m 47s
INFO:tensorflow:Oracle triggered exit
{'regularizer': 'l2', 'reg_strength': 0.0001, 'units': 64, 'activation': 'tanh', 'optimizer': 'adam', 'tuner/epochs': 7, 'tuner/initial_epoch': 0, 'tuner/bracket': 1, 'tuner/round': 0}


**(d)** Test your sentiment-classifier on the test set.

In [8]:
loss, accuracy = best_model.evaluate(test_vec.toarray(), y_test)
print(f'test accuracy - {accuracy:.4f}')

157/157 [==============================] - 0s 817us/step - loss: 0.3106 - accuracy: 0.8882
test accuracy - 0.8882


**(e)** Use the classifier to classify a few sentences you write yourselves. 

In [7]:
pos_review = "I really love Interstellar. Hans Zimmer is the best choice of composer, and the black hole simulation is beautiful."
neg_review = "I hate how the Godfather series dropped off after the first movie. Worst quality dip I have ever seen."
ambiguous_review = "Dune parts I and II are very intriguing works, but they stray from the books, which I feel is cheating. It makes up for it with the soundtrack."

ratings = [1, 0, 1]
custom_reviews = [pos_review, neg_review, ambiguous_review]

custom_vec = vectorizer.transform(custom_reviews)

predictions = best_model.predict(custom_vec.toarray()) # probabilities
predicted_labels = (predictions > 0.5).astype(int)

print(f"probabilities - \n{predictions}\n")
print(f"labels - \n{predicted_labels}\n")

1/1 [==============================] - 0s 66ms/step
probabilities - 
[[0.8111904 ]
 [0.26124394]
 [0.7164021 ]]

labels - 
[[1]
 [0]
 [1]]

